In [333]:
import pandas as pd
import numpy as np
from sqlalchemy import create_engine
import ast

In [359]:
df = pd.read_csv('movies.csv') 
df = df[['id', 'title', 'budget', 'revenue', 'genres', 'belongs_to_collection', 
          'release_date', 'vote_average', 'vote_count', 'runtime']]
df = df[(df['budget']>0) & (df['revenue']>0)] #remove any rows where either budget are 0/unknown 


<class 'pandas.DataFrame'>
Index: 4991 entries, 0 to 9994
Data columns (total 10 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   id                     4991 non-null   int64  
 1   title                  4991 non-null   str    
 2   budget                 4991 non-null   float64
 3   revenue                4991 non-null   int64  
 4   genres                 4991 non-null   str    
 5   belongs_to_collection  1632 non-null   str    
 6   release_date           4991 non-null   str    
 7   vote_average           4991 non-null   float64
 8   vote_count             4991 non-null   int64  
 9   runtime                4991 non-null   int64  
dtypes: float64(2), int64(4), str(4)
memory usage: 428.9 KB


In [360]:
#clean up genres, extract just name

def extract_genres(genres_str):
    genres = ast.literal_eval(genres_str) #convert to dictionary
    return [g['name'] for g in genres]

df['genres'] = df['genres'].apply(extract_genres)


In [361]:
#extract name only from collection
def extract_collection(collection_str):
    if pd.isna(collection_str):
        return None
    else:
        collection_dict = ast.literal_eval(collection_str) #convert to dictionary
        return collection_dict['name']
df['collection'] = df['belongs_to_collection'].apply(extract_collection)



In [362]:

#belongs_to_collection true/false 
df['belongs_to_collection'] = df['belongs_to_collection'].notna()
df = df.rename(columns={'belongs_to_collection': 'is_franchise'})


In [363]:
#return on investment for each movie
df['roi'] = (df['revenue'] - df['budget'])/df['budget']

,id,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,collection,roi
0,1011477,Karate Kid: Legends,45000000.0,104560790,"[Action, Adventure, Drama]",True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573
1,541671,Ballerina,90000000.0,131611905,"[Action, Thriller, Crime]",True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355
2,1061474,Superman,225000000.0,217000000,"[Science Fiction, Adventure, Action]",False,2025-07-09,7.470,538,130,NaN,-0.035556
7,1234821,Jurassic World Rebirth,180000000.0,529463000,"[Science Fiction, Adventure, Action]",True,2025-07-01,6.400,566,134,Jurassic Park Collection,1.941461
8,986056,Thunderbolts*,180000000.0,382027956,"[Action, Science Fiction, Adventure]",False,2025-04-30,7.428,1767,127,NaN,1.122378
10,574475,Final Destination Bloodlines,50000000.0,285153000,"[Horror, Mystery]",True,2025-05-14,7.200,1605,110,Final Destination Collection,4.703060
12,552524,Lilo & Stitch,100000000.0,994264677,"[Family, Science Fiction, Comedy, Adventure]",True,2025-05-17,7.154,827,108,Lilo & Stitch (Live-Action) Collection,8.942647
14,1087192,How to Train Your Dragon,150000000.0,560773000,"[Fantasy, Family, Action]",True,2025-06-06,7.887,630,125,How to Train Your Dragon (Live-Action) Collection,2.738487
15,1151031,Bring Her Back,15000000.0,22878745,[Horror],False,2025-05-28,7.425,242,104,NaN,0.525250
16,911430,F1,200000000.0,393395000,"[Action, Drama]",False,2025-06-25,7.664,727,156,NaN,0.966975


In [364]:
#remove duplicates, many are remakes so drop exact duplicates only (same release date)
df['title'].duplicated().sum() #172 duplicates
df = df.drop_duplicates(subset=['title', 'release_date']) 
df['release_date'] = pd.to_datetime(df['release_date']) 

In [365]:
df_genres = df.explode('genres')


In [366]:
#create new franchise_status dataframe with SQL: -- distinguish franchise status from starter, subsequent and non-franchise 
franchise_status_df = pd.read_sql('''
WITH ranking_collection AS (SELECT id, title, release_date, collection, is_franchise, 
RANK() OVER (PARTITION BY collection ORDER BY release_date) AS rank
FROM movies)

SELECT id,  
CASE 
WHEN is_franchise = True AND rank = 1 THEN 'starter' 
WHEN is_franchise = True AND rank > 1 THEN 'subsequent' 
ELSE 'standalone'
END AS franchise_status
FROM ranking_collection;
''', engine)

In [367]:
#merge with original dataframe 
franchise_status_df.head()
df = df.merge(franchise_status_df, how='inner', on='id')
df.head(10)

,id,title,budget,revenue,genres,is_franchise,release_date,vote_average,vote_count,runtime,collection,roi,franchise_status
0,1011477,Karate Kid: Legends,45000000.0,104560790,"[Action, Adventure, Drama]",True,2025-05-08,7.281,367,94,The Karate Kid Collection,1.323573,subsequent
1,541671,Ballerina,90000000.0,131611905,"[Action, Thriller, Crime]",True,2025-06-04,7.453,944,125,Ballerina Collection,0.462355,starter
2,1061474,Superman,225000000.0,217000000,"[Science Fiction, Adventure, Action]",False,2025-07-09,7.470,538,130,NaN,-0.035556,standalone
3,1234821,Jurassic World Rebirth,180000000.0,529463000,"[Science Fiction, Adventure, Action]",True,2025-07-01,6.400,566,134,Jurassic Park Collection,1.941461,subsequent
4,986056,Thunderbolts*,180000000.0,382027956,"[Action, Science Fiction, Adventure]",False,2025-04-30,7.428,1767,127,NaN,1.122378,standalone
5,574475,Final Destination Bloodlines,50000000.0,285153000,"[Horror, Mystery]",True,2025-05-14,7.200,1605,110,Final Destination Collection,4.703060,subsequent
6,552524,Lilo & Stitch,100000000.0,994264677,"[Family, Science Fiction, Comedy, Adventure]",True,2025-05-17,7.154,827,108,Lilo & Stitch (Live-Action) Collection,8.942647,starter
7,1087192,How to Train Your Dragon,150000000.0,560773000,"[Fantasy, Family, Action]",True,2025-06-06,7.887,630,125,How to Train Your Dragon (Live-Action) Collection,2.738487,starter
8,1151031,Bring Her Back,15000000.0,22878745,[Horror],False,2025-05-28,7.425,242,104,NaN,0.525250,standalone
9,911430,F1,200000000.0,393395000,"[Action, Drama]",False,2025-06-25,7.664,727,156,NaN,0.966975,standalone


In [ ]:
engine = create_engine('postgresql://localhost/movies_db')
df.to_sql('movies', con=engine, if_exists='replace', index=False)
df_genres.to_sql(name='movie_genres', con=engine, if_exists='replace', index=False)

662

In [ ]:
df.info()

<class 'pandas.DataFrame'>
Index: 4953 entries, 0 to 9994
Data columns (total 10 columns):
 #   Column        Non-Null Count  Dtype         
---  ------        --------------  -----         
 0   title         4953 non-null   str           
 1   budget        4953 non-null   float64       
 2   revenue       4953 non-null   int64         
 3   genres        4953 non-null   object        
 4   is_franchise  4953 non-null   bool          
 5   release_date  4953 non-null   datetime64[us]
 6   vote_average  4953 non-null   float64       
 7   vote_count    4953 non-null   int64         
 8   runtime       4953 non-null   int64         
 9   roi           4953 non-null   float64       
dtypes: bool(1), datetime64[us](1), float64(3), int64(3), object(1), str(1)
memory usage: 391.8+ KB
